# Structure to 13C NMR

Predict chemical shifts for the selected nucleus from a molecular structure, using existing frozen weights where available.

This notebook displays recorded real evaluation results. Model inference is opt-in and never trains parameters.

In [1]:
import json, os, sys, subprocess, tempfile
from pathlib import Path
root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'openmolspec_bench').is_dir()), None)
if root is None:
    raise RuntimeError('Run this notebook from the cloned OpenMolSpec frontend repository')
task_id = 'nmr.forward.13C'
example_input = {'smiles': 'CCO', 'formula': 'C2H6O', 'h_nmr': [{'type': '1H NMR', 'chemical_shift': '1.20 (t, 3H), 3.65 (q, 2H)'}], 'c_nmr': [{'type': '13C NMR', 'chemical_shift': '18.2, 58.3'}], 'modality_mask': [1, 1]}
print(json.dumps(example_input, indent=2, ensure_ascii=False))

{
  "smiles": "CCO",
  "formula": "C2H6O",
  "h_nmr": [
    {
      "type": "1H NMR",
      "chemical_shift": "1.20 (t, 3H), 3.65 (q, 2H)"
    }
  ],
  "c_nmr": [
    {
      "type": "13C NMR",
      "chemical_shift": "18.2, 58.3"
    }
  ],
  "modality_mask": [
    1,
    1
  ]
}


## Recorded evaluation

Results retain their sample size, conditions and source file; an execution pass does not imply accuracy.

In [2]:
index = json.loads((root / 'documentation/assets/evaluation_index.json').read_text())
recorded = index.get(task_id, [])
print(json.dumps(recorded or {'status': 'not_evaluated', 'reason': 'Required compatible assets or adapter unavailable'}, indent=2, ensure_ascii=False))

[
  {
    "task": "nmr.forward.13C",
    "dataset_name": "nmrspec_test",
    "status": "passed",
    "selection": {
      "read": 32,
      "missing_modality": 0,
      "duplicate_structure": 0,
      "exceeds_model_capacity": 0
    },
    "source_file": "smoke_forward_20261115_v3.json",
    "sample_limit": 32,
    "models": [
      {
        "model": "nmrnet",
        "metrics": {
          "matched_peak_mae_ppm": 1.166839493503614,
          "matched_peak_rmse_ppm": 2.3898515362116135,
          "peak_count_mae": 2.65625,
          "matched_fraction": 0.8683153839313926,
          "examples": 32,
          "scored_examples": 32,
          "aggregation": "equal_weight_per_molecule",
          "assignment": "minimum_absolute_error_without_atom_labels"
        }
      },
      {
        "model": "nmrnetpp",
        "metrics": {
          "matched_peak_mae_ppm": 0.8163807039222384,
          "matched_peak_rmse_ppm": 2.0137184870551987,
          "peak_count_mae": 2.65625,
          "matc

## Optional frozen inference

Set `OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE=1` and configure the workspace and model runtime. Missing weights cause a skip/error, never training.

In [3]:
if os.environ.get('OPENMOLSPEC_RUN_NOTEBOOK_INFERENCE') == '1':
    runtime = os.environ.get('OPENMOLSPEC_PYTHON', sys.executable)
    workspace = os.environ.get('OPENMOLSPEC_BENCH_ROOT', str(root.parent))
    with tempfile.TemporaryDirectory(prefix='openmolspec-notebook-') as directory:
        command = [runtime, '-m', 'openmolspec_bench', 'run', '--workspace', workspace, '--tasks', task_id, '--limit', '2', '--output', str(Path(directory) / 'result.json')]
        subprocess.run(command, cwd=root, check=True)
        print((Path(directory) / 'result.json').read_text())
else:
    print('Model execution disabled; recorded evaluation displayed above. No training performed.')

Model execution disabled; recorded evaluation displayed above. No training performed.
